In [1]:
import os
if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
# tahtan
# Import Sionna
import sys
sys.path.append('../')
import sionna

# try:
#     import sionna
# except ImportError as e:
#     # Install Sionna if package is not already installed
#     import os
#     os.system("pip install sionna")
#     import sionna

import tensorflow as tf
# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

sionna.config.seed = 42 # Set seed for reproducible results

# Load the required Sionna components
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver, CarrierConfig, PUSCHDMRSConfig,\
                        TBConfig, PUSCHPilotPattern, TBEncoder, PUSCHPrecoder, LayerMapper, LayerDemapper, check_pusch_configs,\
                        TBDecoder, PUSCHLSChannelEstimator
from sionna.nr.utils import generate_prng_seq
from sionna.channel import AWGN, RayleighBlockFading, OFDMChannel, TimeChannel, time_lag_discrete_time_channel
from sionna.channel.utils import * 
from sionna.channel.tr38901 import Antenna, AntennaArray, UMi, UMa, RMa, TDL, CDL
from sionna.channel import gen_single_sector_topology as gen_topology
from sionna.utils import compute_ber, ebnodb2no, sim_ber, array_to_hash, create_timestamped_folders, b2b, f2f, BinarySource
from sionna.ofdm import KBestDetector, LinearDetector, MaximumLikelihoodDetector,\
        LSChannelEstimator, LMMSEEqualizer, RemoveNulledSubcarriers, ResourceGridDemapper,\
        ResourceGrid, ResourceGridMapper, OFDMModulator
from sionna.mimo import StreamManagement
from sionna.mapping import Mapper, Demapper

from sionna.nr.my_abc import *

ImportError: cannot import name 'Model' from 'tensorflow.keras.layers' (/workspaces/thanh/.venv/lib/python3.11/site-packages/keras/api/_v2/keras/layers/__init__.py)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from datetime import datetime, timedelta
# from bs4 import BeautifulSoup
import pickle
from collections import namedtuple
import json
from tqdm.notebook import tqdm
import itertools
import io
import h5py

## A Hello World Example

Let us start with a simple "Hello, World!" example in which we will simulate PUSCH transmissions from a single transmitter to a single receiver over an AWGN channel.

In [ ]:
def data_loader(df, dir, saved_dataset='hdf5'):
    assert saved_dataset in ['hdf5', 'pickle'], "saved data set should be 'pickle' or 'hdf5'."
    assert df['nTBSize'].nunique() == 1, "Not all elements have the same TB size."
    assert df['Dmrs_mask'].map(len).nunique() == 1, "Not all elements have the same number of Dmrs/Data symbols."
    for pusch_record in df.itertuples():
        data_filename = pusch_record.Data_filename
        data_dirname = pusch_record.Data_dirname
        esno_db = pusch_record.Esno_db
        index = pusch_record.Index
        data_mask = ~pusch_record.Dmrs_mask
        if saved_dataset == 'hdf5':
            b,c,y = load_hdf5(f'{dir}/{data_dirname}', data_filename)
        else:
            b,c,y = load_pickle(f'{dir}/{data_dirname}', data_filename)
        data_ind = tf.where(
            tf.repeat(
                data_mask.reshape(y.shape[-2],y.shape[-1],1),
                len(c)/np.sum(data_mask, axis=-1),
                axis=2)
            )
        yield index, esno_db, c, y, b, data_ind

def preprocessing(index, esno_db, c, y, b, data_ind):
    
    return index, esno_db, c, y, b, data_ind

dataset_dir = f'../Pusch_data/dataset'
pickles_dir = f'{dataset_dir}/pickle'
hdf5_dir = f'{dataset_dir}/hdf5'
parquet_dir = f'{dataset_dir}/parquet'

# parquet_name = 20250306015239576970
parquet_name = 20250306015626321031

In [44]:
df = pd.read_parquet(f'{parquet_dir}/{parquet_name}.parquet', engine="pyarrow")

In [ ]:
cfg_cols = ['nPhyCellId', 'nCpType', 'nSubcSpacing', 'nBWPSize', 'nBWPStart', 'nSlot',
'nDMRSConfigType', 'nNrOfDMRSSymbols', 'nDMRSAddPos', 'nPortIndex', 'nNIDnSCID', 'nSCID', 'nNrOfCDMs', 'nDMRSTypeAPos',
'nNid', 'nMcsTable', 'nMCS',
'nMappingType', 'nNrOfLayers', 'nTransmissionScheme', 'nPMI', 'nTransPrecode', 'nRNTI',
'nStartSymbolIndex', 'nNrOfSymbols', 'nRBStart', 'nRBSize',
'nPTRSPresent', 'nAck', 'nCsiPart1', 'nCsiPart2', 'nTpPi2BPSK']

In [59]:
df_copy = df[cfg_cols].copy()
for col in df_copy.columns:
    if isinstance(df_copy[col].iloc[0], np.ndarray):  # Check first element type
        df_copy[col] = df_copy[col].apply(lambda x: tuple(x) if isinstance(x, np.ndarray) else x)
df_drop_dup = df_copy.drop_duplicates()
df_drop_dup

,nPhyCellId,nCpType,nSubcSpacing,nBWPSize,nBWPStart,nSlot,nDMRSConfigType,nNrOfDMRSSymbols,nDMRSAddPos,nPortIndex,...,nRNTI,nStartSymbolIndex,nNrOfSymbols,nRBStart,nRBSize,nPTRSPresent,nAck,nCsiPart1,nCsiPart2,nTpPi2BPSK
0,246,0,1,273,0,4,0,1,1,"(0,)",...,20004,0,14,0,273,0,0,0,0,0


In [60]:
grouped_indices = df_copy.reset_index().groupby(list(df_copy.columns)).agg(indices=('index',list)).reset_index()
grouped_indices.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 33 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   nPhyCellId           1 non-null      int64 
 1   nCpType              1 non-null      int64 
 2   nSubcSpacing         1 non-null      int64 
 3   nBWPSize             1 non-null      int64 
 4   nBWPStart            1 non-null      int64 
 5   nSlot                1 non-null      int64 
 6   nDMRSConfigType      1 non-null      int64 
 7   nNrOfDMRSSymbols     1 non-null      int64 
 8   nDMRSAddPos          1 non-null      int64 
 9   nPortIndex           1 non-null      object
 10  nNIDnSCID            1 non-null      int64 
 11  nSCID                1 non-null      int64 
 12  nNrOfCDMs            1 non-null      int64 
 13  nDMRSTypeAPos        1 non-null      int64 
 14  nNid                 1 non-null      int64 
 15  nMcsTable            1 non-null      int64 
 16  nMCS        

In [61]:
grouped_indices

,nPhyCellId,nCpType,nSubcSpacing,nBWPSize,nBWPStart,nSlot,nDMRSConfigType,nNrOfDMRSSymbols,nDMRSAddPos,nPortIndex,...,nStartSymbolIndex,nNrOfSymbols,nRBStart,nRBSize,nPTRSPresent,nAck,nCsiPart1,nCsiPart2,nTpPi2BPSK,indices
0,246,0,1,273,0,4,0,1,1,"(0,)",...,0,14,0,273,0,0,0,0,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,..."


In [57]:
def create_configs_from_df(grouped_df: pd.DataFrame) -> List[MyConfig]:
    configs_and_indices = []
    
    for row in grouped_df.itertuples():
        sys_cfg = SystemConfig(
            NCellId=row.nPhyCellId if hasattr(row, 'nPhyCellId') else row.nNid,
            CpType=row.nCpType,
            NTxAnt=1,  # Keeping it constant as in the default
            NRxAnt=8,  # Keeping it constant as in the default
            BwpNRb=row.nBWPSize,
            BwpRbOffset=row.nBWPStart
        )

        ue_cfg = UeConfig(
            TransformPrecoding=row.nTransPrecode,
            Rnti=row.nRNTI,
            nId=row.nNid,
            NLayers=row.nNrOfLayers,
            FirstSymb=row.nStartSymbolIndex,
            NPuschSymbAll=row.nNrOfSymbols,
            FirstPrb=row.nRBStart,
            NPrb=row.nRBSize,
            McsTable=row.nMcsTable,
            Mcs=row.nMCS,
            nScId=row.nSCID,
            NnScIdId=row.nNIDnSCID,
            DmrsConfigurationType=row.nDMRSConfigType,
            DmrsDuration=row.nNrOfDMRSSymbols,
            DmrsAdditionalPosition=row.nDMRSAddPos,
            PuschMappingType=row.nMappingType,
            DmrsTypeAPosition=row.nDMRSTypeAPos,
            Ptrs=row.nPTRSPresent,
            OAck=row.nAck,
            OCsi1=row.nCsiPart1,
            OCsi2=row.nCsiPart2,
            TpPi2Bpsk=row.nTpPi2BPSK
        )

        my_cfg = MyConfig(Sys=sys_cfg, Ue=[ue_cfg])
        pusch_cfg = MyPUSCHConfig(my_cfg, row.nSlot, row.nSFN)
        configs_and_indices.append((pusch_cfg, row.indices))

    return configs_and_indices

configs_and_indices = create_configs_from_df(grouped_indices)

AttributeError: 'Pandas' object has no attribute 'nPTRSPresent'

In [34]:
dataset_0 = tf.data.Dataset.from_generator(
            lambda: data_loader(df.iloc[configs_and_indices[1][1]], hdf5_dir),
            output_types=(tf.int32, tf.float32, tf.float32, tf.complex64, tf.float32, tf.int32))

In [35]:
for n, (index, esno_db, c, y, b, data_ind) in enumerate(dataset_0.map(preprocessing).batch(2)):
    print(index, n, esno_db.shape, c.shape, y.shape, b.shape, data_ind.shape)

tf.Tensor([ 1 13], shape=(2,), dtype=int32) 0 (2,) (2, 1152) (2, 8, 14, 48) (2, 288) (2, 1152, 3)
tf.Tensor([25 37], shape=(2,), dtype=int32) 1 (2,) (2, 1152) (2, 8, 14, 48) (2, 288) (2, 1152, 3)
tf.Tensor([49 61], shape=(2,), dtype=int32) 2 (2,) (2, 1152) (2, 8, 14, 48) (2, 288) (2, 1152, 3)
tf.Tensor([73 85], shape=(2,), dtype=int32) 3 (2,) (2, 1152) (2, 8, 14, 48) (2, 288) (2, 1152, 3)
tf.Tensor([ 97 109], shape=(2,), dtype=int32) 4 (2,) (2, 1152) (2, 8, 14, 48) (2, 288) (2, 1152, 3)
tf.Tensor([121 133], shape=(2,), dtype=int32) 5 (2,) (2, 1152) (2, 8, 14, 48) (2, 288) (2, 1152, 3)
tf.Tensor([145 157], shape=(2,), dtype=int32) 6 (2,) (2, 1152) (2, 8, 14, 48) (2, 288) (2, 1152, 3)
tf.Tensor([169 181], shape=(2,), dtype=int32) 7 (2,) (2, 1152) (2, 8, 14, 48) (2, 288) (2, 1152, 3)
tf.Tensor([193 205], shape=(2,), dtype=int32) 8 (2,) (2, 1152) (2, 8, 14, 48) (2, 288) (2, 1152, 3)
tf.Tensor([217 229], shape=(2,), dtype=int32) 9 (2,) (2, 1152) (2, 8, 14, 48) (2, 288) (2, 1152, 3)
tf.Tenso

In [36]:
ue_antenna = Antenna(polarization="single",
                polarization_type="V",
                antenna_pattern="38.901",
                carrier_frequency=2.55e9)

gnb_array = AntennaArray(num_rows=1,
                        num_cols=8//2,
                        polarization="dual",
                        polarization_type="cross",
                        antenna_pattern="38.901",
                        carrier_frequency=2.55e9)

channel_model = CDL(model = 'C',
                            delay_spread = 150*1e-9,
                            carrier_frequency = 2.55e9,
                            ut_array = ue_antenna,
                            bs_array = gnb_array,
                            direction = 'uplink',
                            min_speed = 10,
                            max_speed = 10)

simulator = MySimulator(configs_and_indices[1][0])

channel = OFDMChannel(channel_model=channel_model, resource_grid=simulator.resource_grid,
                                    add_awgn=False, normalize_channel=True, return_channel=True)

b, c, y = simulator.sim(1, channel, 1.)
print(y.shape)
simulator.rec(y)

(1, 1, 8, 14, 48)


(<tf.Tensor: shape=(1, 1, 288), dtype=float32, numpy=
 array([[[1., 1., 0., 0., 0., 1., 1., 1., 0., 1., 1., 0., 0., 1., 1., 0.,
          0., 1., 0., 1., 1., 1., 0., 1., 0., 1., 0., 1., 1., 0., 0., 1.,
          0., 0., 1., 0., 1., 1., 0., 1., 0., 0., 0., 0., 1., 0., 0., 0.,
          1., 0., 1., 1., 0., 0., 0., 1., 1., 1., 1., 0., 0., 0., 1., 0.,
          0., 1., 1., 0., 1., 1., 0., 1., 1., 0., 1., 0., 0., 0., 0., 0.,
          0., 0., 0., 1., 0., 0., 0., 0., 0., 1., 1., 0., 0., 1., 0., 1.,
          1., 0., 1., 1., 0., 1., 0., 0., 0., 0., 0., 1., 1., 1., 0., 1.,
          0., 1., 1., 1., 0., 1., 1., 0., 0., 1., 1., 1., 0., 1., 1., 0.,
          1., 0., 1., 0., 1., 1., 1., 0., 1., 0., 0., 0., 1., 0., 1., 0.,
          1., 1., 1., 0., 1., 1., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
          0., 0., 0., 0., 0., 1., 0., 0., 1., 0., 1., 0., 1., 1., 1., 0.,
          0., 1., 0., 0., 0., 1., 1., 1., 1., 0., 0., 0., 1., 0., 0., 0.,
          1., 0., 0., 1., 1., 1., 0., 1., 0., 1., 1., 1., 

In [37]:
for (config, indices) in configs_and_indices:
    print(indices)

[0, 12, 24, 36, 48, 60, 72, 84, 96, 108, 120, 132, 144, 156, 168, 180, 192, 204, 216, 228, 240, 252, 264, 276, 288, 300, 312, 324, 336, 348, 360, 372, 384, 396, 408, 420, 432, 444, 456, 468, 480, 492]
[1, 13, 25, 37, 49, 61, 73, 85, 97, 109, 121, 133, 145, 157, 169, 181, 193, 205, 217, 229, 241, 253, 265, 277, 289, 301, 313, 325, 337, 349, 361, 373, 385, 397, 409, 421, 433, 445, 457, 469, 481, 493]
[2, 14, 26, 38, 50, 62, 74, 86, 98, 110, 122, 134, 146, 158, 170, 182, 194, 206, 218, 230, 242, 254, 266, 278, 290, 302, 314, 326, 338, 350, 362, 374, 386, 398, 410, 422, 434, 446, 458, 470, 482, 494]
[3, 15, 27, 39, 51, 63, 75, 87, 99, 111, 123, 135, 147, 159, 171, 183, 195, 207, 219, 231, 243, 255, 267, 279, 291, 303, 315, 327, 339, 351, 363, 375, 387, 399, 411, 423, 435, 447, 459, 471, 483, 495]
[4, 16, 28, 40, 52, 64, 76, 88, 100, 112, 124, 136, 148, 160, 172, 184, 196, 208, 220, 232, 244, 256, 268, 280, 292, 304, 316, 328, 340, 352, 364, 376, 388, 400, 412, 424, 436, 448, 460, 472, 484,

In [38]:
simulator_0 = MySimulator(configs_and_indices[0][0])
for (config, indices) in configs_and_indices[1:2]:
    dataset_0 = tf.data.Dataset.from_generator(
                lambda: data_loader(df.iloc[indices], hdf5_dir),
                output_types=(tf.int32, tf.float32, tf.float32, tf.complex64, tf.float32, tf.int32))
    
    for n, (index, esno_db, c, y, b, data_ind) in enumerate(dataset_0.map(preprocessing).batch(1)):
        _,_, crc = simulator_0.rec(y[:,None])
        print(n, index, crc, esno_db)

0 tf.Tensor([1], shape=(1,), dtype=int32) tf.Tensor([[False]], shape=(1, 1), dtype=bool) tf.Tensor([-5.], shape=(1,), dtype=float32)
1 tf.Tensor([13], shape=(1,), dtype=int32) tf.Tensor([[ True]], shape=(1, 1), dtype=bool) tf.Tensor([-4.5], shape=(1,), dtype=float32)
2 tf.Tensor([25], shape=(1,), dtype=int32) tf.Tensor([[ True]], shape=(1, 1), dtype=bool) tf.Tensor([-4.], shape=(1,), dtype=float32)
3 tf.Tensor([37], shape=(1,), dtype=int32) tf.Tensor([[ True]], shape=(1, 1), dtype=bool) tf.Tensor([-3.5], shape=(1,), dtype=float32)
4 tf.Tensor([49], shape=(1,), dtype=int32) tf.Tensor([[ True]], shape=(1, 1), dtype=bool) tf.Tensor([-3.], shape=(1,), dtype=float32)
5 tf.Tensor([61], shape=(1,), dtype=int32) tf.Tensor([[ True]], shape=(1, 1), dtype=bool) tf.Tensor([-2.5], shape=(1,), dtype=float32)
6 tf.Tensor([73], shape=(1,), dtype=int32) tf.Tensor([[ True]], shape=(1, 1), dtype=bool) tf.Tensor([-2.], shape=(1,), dtype=float32)
7 tf.Tensor([85], shape=(1,), dtype=int32) tf.Tensor([[ True

KeyboardInterrupt: 

In [ ]:
configs_and_indices[0][1]

In [26]:
for n, (index, esno_db, c, y, b, data_ind) in enumerate(dataset.map(preprocessing).batch(2)):
    print(index, n, esno_db.shape, c.shape, y.shape, b.shape, data_ind.shape)
    print(simulator.rec(y[:,None]))

tf.Tensor([13 14], shape=(2,), dtype=int32) 0 (2,) (2, 78624) (2, 8, 14, 3276) (2, 19464) (2, 78624, 3)
(<tf.Tensor: shape=(2, 1, 19464), dtype=float32, numpy=
array([[[0., 1., 1., ..., 1., 1., 0.]],

       [[1., 1., 0., ..., 1., 1., 1.]]], dtype=float32)>, <tf.Tensor: shape=(2, 1, 1, 78624), dtype=float32, numpy=
array([[[[ 35599.652,  28177.914, -54362.188, ...,  20449.74 ,
           79261.234, -94995.42 ]]],


       [[[-52336.746, -34549.38 ,  76954.39 , ..., -36116.195,
           -9920.474,  15615.282]]]], dtype=float32)>, <tf.Tensor: shape=(2, 1), dtype=bool, numpy=
array([[False],
       [False]])>)
tf.Tensor([16 10], shape=(2,), dtype=int32) 1 (2,) (2, 78624) (2, 8, 14, 3276) (2, 19464) (2, 78624, 3)
(<tf.Tensor: shape=(2, 1, 19464), dtype=float32, numpy=
array([[[0., 0., 0., ..., 1., 0., 0.]],

       [[0., 1., 0., ..., 1., 1., 1.]]], dtype=float32)>, <tf.Tensor: shape=(2, 1, 1, 78624), dtype=float32, numpy=
array([[[[ 31416.582 ,  29704.193 , -74879.555 , ...,    721.2676,

In [20]:
from tensorflow.keras.layers import Layer, Conv2D, LayerNormalization, SeparableConv2D
from tensorflow.nn import relu
class ResidualBlock(tf.keras.Model):
    r"""
    This Keras layer implements a convolutional residual block made of two convolutional layers with ReLU activation, layer normalization, and a skip connection.
    The number of convolutional channels of the input must match the number of kernel of the convolutional layers ``num_conv_channel`` for the skip connection to work.

    Input
    ------
    : [batch size, num time samples, num subcarriers, num_conv_channel], tf.float
        Input of the layer

    Output
    -------
    : [batch size, num time samples, num subcarriers, num_conv_channel], tf.float
        Output of the layer
    """

    def build(self, input_shape):

        # Layer normalization is done over the last three dimensions: time, frequency, conv 'channels'
        self._layer_norm_1 = LayerNormalization(axis=(-1, -2, -3))
        self._conv_1 = SeparableConv2D(filters= 64,
                              kernel_size=[3,3],
                              padding='same',
                              activation=None)
        # Layer normalization is done over the last three dimensions: time, frequency, conv 'channels'
        self._layer_norm_2 = LayerNormalization(axis=(-1, -2, -3))
        self._conv_2 = SeparableConv2D(filters= 128,
                              kernel_size=[3,3],
                              padding='same',
                              activation=None)

    def call(self, inputs):
        z = self._layer_norm_1(inputs)
        z = relu(z)
        z = self._conv_1(z)
        z = self._layer_norm_2(z)
        z = relu(z)
        z = self._conv_2(z) # [batch size, num time samples, num subcarriers, num_channels]
        # Skip connection
        z = z + inputs

        return z

class CustomNeuralReceiver(tf.keras.Model):
    r"""
    Keras layer implementing a residual convolutional neural receiver.

    This neural receiver is fed with the post-DFT received samples, forming a resource grid of size num_of_symbols x fft_size, and computes LLRs on the transmitted coded bits.
    These LLRs can then be fed to an outer decoder to reconstruct the information bits.

    Input
    ------
    y_no: [batch size, num ofdm symbols, num subcarriers, 2*num rx antenna + 1], tf.float32
        Concatenated received samples and noise variance.
(
    y : [batch size, num rx antenna, num ofdm symbols, num subcarriers], tf.complex
        Received post-DFT samples.

    no : [batch size], tf.float32
        Noise variance. At training, a different noise variance value is sampled for each batch example.
)
    Output
    -------
    : [batch size, num ofdm symbols, num subcarriers, num_bits_per_symbol]
        LLRs on the transmitted bits.
    """

    def __init__(self, training = False):
        super(CustomNeuralReceiver, self).__init__()
        self._training = training

    def build(self, input_shape):

        # Input convolution
        self._input_conv = Conv2D(filters= 128,
                                  kernel_size=[3,3],
                                  padding='same',
                                  activation=None)
        # Residual blocks
        self._res_block_1 = ResidualBlock()
        self._res_block_2 = ResidualBlock()
        self._res_block_3 = ResidualBlock()
        self._res_block_4 = ResidualBlock()
        # Output conv
        self._output_conv = Conv2D(filters= 2,    # QPSK
                                   kernel_size=[3,3],
                                   padding='same',
                                   activation=None)
        

    @tf.function(jit_compile=True)
    def call(self, inputs):
        # Input conv
        z = self._input_conv(inputs)
        # Residual blocks
        z = self._res_block_1(z)
        z = self._res_block_2(z)
        z = self._res_block_3(z)
        z = self._res_block_4(z)
        # Output conv
        z = self._output_conv(z)
        # if self._training == False:
        #     z = tf.cast(z * (2**7), tf.int8)
        return z
    

_model = CustomNeuralReceiver(training = False)
inputs = tf.zeros([1,3276,14,16])
_model(inputs)
_model.summary()

def load_weights(model, pretrained_weights_path):
    # Build Model with random input
    # Load weights
  with open(pretrained_weights_path, 'rb') as f:
    weights = pickle.load(f)
    model.set_weights(weights)
    print(f"Loaded pretrained weights from {pretrained_weights_path}")

#load_weights(_model, '/content/drive/MyDrive/Pusch_data/Model_weights/model_weight_FULL_RB_epoch_40.pkl')
load_weights(_model, '../model_weight_FULL_RB_epoch_40.pkl')


Model: "custom_neural_receiver_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_2 (Conv2D)           multiple                  18560     
                                                                 
 residual_block_4 (Residual  multiple                  17630080  
 Block)                                                          
                                                                 
 residual_block_5 (Residual  multiple                  17630080  
 Block)                                                          
                                                                 
 residual_block_6 (Residual  multiple                  17630080  
 Block)                                                          
                                                                 
 residual_block_7 (Residual  multiple                  17630080  
 Block)                                   

In [21]:
def loss_cal(pred, label):
  bce = tf.nn.sigmoid_cross_entropy_with_logits(label, pred)
  bce = tf.reduce_mean(bce)
  # rate = tf.constant(1.0, tf.float32) - bce/tf.math.log(2.)
  # loss = -rate
  loss = bce
  return loss

In [22]:
df.iloc[0]

nSFN                                                                        0
nSlot                                                                       4
nPDU                                                                        1
nGroup                                                                      1
nUlsch                                                                      1
nUlcch                                                                      0
nRachPresent                                                                0
nRNTI                                                                   20002
nUEId                                                                       0
nBWPSize                                                                  273
nBWPStart                                                                   0
nSubcSpacing                                                                1
nCpType                                                         

In [23]:
y_ = tf.concat([tf.math.real(y), tf.math.imag(y)], axis=1)
y_ = tf.transpose(y_, perm=[0,3,2,1])
y_

<tf.Tensor: shape=(2, 3276, 14, 16), dtype=float32, numpy=
array([[[[ 3.4856141e-01,  6.7655712e-01,  1.6445942e+00, ...,
          -9.4151247e-01, -9.4782925e-01, -1.6929244e+00],
         [-2.0591171e+00,  9.5540023e-01, -5.9710979e-01, ...,
           1.3324263e+00,  2.1872373e+00,  8.1651664e-01],
         [ 8.7811559e-02,  2.6583752e-01,  1.6383942e+00, ...,
           3.7266903e+00,  2.9546466e+00, -2.6750576e-01],
         ...,
         [ 2.7504539e-01, -1.9469850e+00, -1.3311462e+00, ...,
          -3.5341215e+00, -2.4836531e+00, -4.9577165e-01],
         [ 1.5518521e+00,  2.2954221e+00, -3.7538311e-01, ...,
           4.4959621e+00,  3.2524533e+00, -3.8194728e-01],
         [ 6.0679269e-01,  1.1169598e+00,  1.6175511e-01, ...,
          -2.4763751e+00, -4.5051634e-01, -1.3107517e+00]],

        [[-3.8474259e+00, -3.5440068e+00, -6.1099517e-01, ...,
          -6.7074794e-01,  1.4358218e+00,  3.9118029e-02],
         [ 3.0597975e+00,  2.8747330e+00,  8.2642007e-01, ...,
        

In [24]:
pred = _model(y_)
pred = tf.transpose(pred, perm=[0,2,1,3])

In [25]:
pred_ = tf.concat([pred[:,0:2], pred[:,3:11], pred[:,12:14]],axis=-3)
pred_.shape

TensorShape([2, 12, 3276, 2])

In [30]:
c_pred = tf.reshape(pred_, [2,-1])
loss_cal(c_pred, c)

<tf.Tensor: shape=(), dtype=float32, numpy=4.028155>

In [27]:
data_ind

<tf.Tensor: shape=(2, 78624, 3), dtype=int32, numpy=
array([[[   0,    0,    0],
        [   0,    0,    1],
        [   0,    1,    0],
        ...,
        [  13, 3274,    1],
        [  13, 3275,    0],
        [  13, 3275,    1]],

       [[   0,    0,    0],
        [   0,    0,    1],
        [   0,    1,    0],
        ...,
        [  13, 3274,    1],
        [  13, 3275,    0],
        [  13, 3275,    1]]], dtype=int32)>

In [28]:
data_extract = tf.gather_nd(pred, data_ind, 1)

In [29]:
loss_cal(data_extract, c)

<tf.Tensor: shape=(), dtype=float32, numpy=4.028155>

In [ ]:
# def predict(model, y, data_ind):
#     if len(y.shape) == 5: y = y[:,0]
#     y = tf.transpose(
#             tf.concat([tf.math.real(y), tf.math.imag(y)], axis=1),
#             [0,3,2,1]
#         )
#     pred = model(y)
#     c_soft = tf.gather_nd(tf.transpose(pred, [0, 2, 1, 3]), data_ind, 1)
#     return c_soft, pred
# c_soft, pred = predict(_model, y, data_ind)

In [48]:
loss_cal(c_soft, c)

<tf.Tensor: shape=(), dtype=float32, numpy=3.796445>

In [49]:
pred.shape

TensorShape([2, 3276, 14, 2])

In [50]:
_pred = tf.concat([pred[...,0:2,:], pred[...,3:11,:], pred[...,12:14,:]],axis=-2)
_pred.shape

TensorShape([2, 3276, 12, 2])

In [51]:
_pred = tf.transpose(_pred, perm=[0,2,1,3])
_pred

<tf.Tensor: shape=(2, 12, 3276, 2), dtype=float32, numpy=
array([[[[ 1.0469391e+01,  5.0095711e+00],
         [-3.1668515e+00, -5.1410632e+00],
         [-8.6160650e+00, -1.0593261e+01],
         ...,
         [-4.7331257e+00, -9.0367508e-01],
         [ 5.1488929e+00, -8.4180794e+00],
         [ 2.1358950e+00, -1.0051928e-01]],

        [[ 6.4932318e+00,  4.9972405e+00],
         [ 5.7338161e+00,  1.0484211e+01],
         [-3.1574194e+00, -1.1338509e+01],
         ...,
         [-8.5004175e-01,  2.1252000e+00],
         [ 9.0362597e+00,  3.7422626e+00],
         [ 2.0270646e+00,  4.4786558e+00]],

        [[-1.0939690e+01, -9.0610247e+00],
         [ 7.6532638e-01,  3.2562792e+00],
         [-1.8934168e+01,  1.3377622e+00],
         ...,
         [ 1.4547210e+00, -1.9695178e+00],
         [-8.4812746e+00,  9.3804913e+00],
         [-1.0330507e+00,  3.0878043e+00]],

        ...,

        [[ 1.3301461e+01,  4.2935681e+00],
         [ 6.9248252e+00, -9.8991070e+00],
         [-1.1186101

In [52]:
c_pred = tf.reshape(_pred, [2,-1])
loss_cal(c_pred, c)

<tf.Tensor: shape=(), dtype=float32, numpy=3.7758174>